# 01 — Data Exploration

Explore raw GTFS and housing data: distributions, coverage, quality issues.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from kg.data.parsers import parse_stops, parse_stop_times, parse_trips, parse_housing, build_lookups
from kg.data.district_mapper import load_districts, map_all_stops, map_all_flats, aggregate_per_district

sns.set_style('whitegrid')
%matplotlib inline

## 1. GTFS Data

In [ ]:
stops_df = parse_stops()
stop_times_df = parse_stop_times()
trips_df = parse_trips()

print(f'Stops: {len(stops_df):,}')
print(f'Stop times: {len(stop_times_df):,}')
print(f'Trips: {len(trips_df):,}')
print(f'Unique routes: {trips_df["route_id"].nunique()}')

stops_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(stops_df['stop_lon'], stops_df['stop_lat'], s=1, alpha=0.3)
axes[0].set_title('Stop Locations')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

stop_to_trips, _ = build_lookups(stop_times_df, trips_df)
trip_counts = pd.Series([len(v) for v in stop_to_trips.values()])
axes[1].hist(trip_counts.clip(upper=trip_counts.quantile(0.95)), bins=50)
axes[1].set_title('Trips per Stop (clipped at 95th pct)')
axes[1].set_xlabel('Number of trips')

plt.tight_layout()
plt.show()

## 2. Housing Data

In [ ]:
housing_df = parse_housing()
print(f'Listings: {len(housing_df):,}')
print(f'Price range: {housing_df["price"].min():.0f} – {housing_df["price"].max():.0f} EUR')
print(f'Size range: {housing_df["size"].min():.0f} – {housing_df["size"].max():.0f} m²')

housing_df.describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(housing_df['price'].clip(upper=housing_df['price'].quantile(0.95)), bins=40)
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price (EUR)')

axes[1].hist(housing_df['price_per_sqm'].clip(upper=housing_df['price_per_sqm'].quantile(0.95)), bins=40)
axes[1].set_title('Price per m²')
axes[1].set_xlabel('EUR/m²')

axes[2].scatter(housing_df['lon'], housing_df['lat'], s=3, alpha=0.3, c=housing_df['price_per_sqm'], cmap='RdYlGn_r')
axes[2].set_title('Listings by Price/m²')
axes[2].set_xlabel('Longitude')

plt.tight_layout()
plt.show()

## 3. District-Level Aggregation

In [ ]:
districts_gdf = load_districts()
stops_with_district = map_all_stops(stops_df, districts_gdf)
housing_with_district = map_all_flats(housing_df, districts_gdf)

agg = aggregate_per_district(stops_with_district, housing_with_district, stop_times_df, trips_df)
agg

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

agg['stop_count'].plot.bar(ax=axes[0, 0], title='Stops per District')
agg['route_count'].plot.bar(ax=axes[0, 1], title='Routes per District')
agg['avg_price'].plot.bar(ax=axes[1, 0], title='Avg. Price per District')
agg['avg_price_per_sqm'].plot.bar(ax=axes[1, 1], title='Avg. Price/m² per District')

for ax in axes.flat:
    ax.set_xlabel('District')

plt.tight_layout()
plt.show()

## 4. Data Quality Notes

- Some stops may fall outside Vienna's district polygons (e.g. terminus stops across the border) — these get `district = NaN` and are excluded from district-level stats.
- Housing data coverage varies by district — districts with few listings will have less reliable price averages.
- Stop IDs in `stop_times.txt` use a longer format than `stops.txt`; truncation to the first 3 colon segments is necessary for matching.